In [ ]:
{"tags": ["remove_input", ]}

## Load OS module
import os

## Add environment variable for R installation
#os.environ['R_HOME'] = r"C:\Program Files\R\R-4.4.1"
os.environ["PATH"] = r"C:\Users\ArPa3547\AppData\Local\Programs\R\R-4.6.0\bin\x64"

# Vignette: healthiar with Python
Using `healthiar` in Python is possible with the dedicated wrapper package. The wrapper consists of a Python program that allows you to use the `healthiar` R package as if it were a Python package. In technical terms, the Python package works by running R embedded in a Python process.

The Python wrapper for `healthiar` is an implementation of the [`rpy2` interface](https://rpy2.github.io/).

## Requirements

To use the Python package, you need R, with the `healthiar` package installed.

## Installation

1.  Install the `healthiar` Python package with `pip`:

In [ ]:
pip install healthiar

Note that this does **not** install the `healthiar` R package.

2.  Add the path of your R installation as environment variable. In Python, you can do this as follows:

In [ ]:
import os

os.environ['R_HOME'] = "/path/to/R/R-4.6.0"
## or
os.environ["PATH"] = "/path/to/R/R-4.6.0/bin/x64"

## Modules
### Data conversion

The `conversion` module contains the functions for converting input and output data, needed for a correct functioning of the wrapper:

- `py_to_r()`: To convert Python input data to the `rpy2` format;

- `r_to_py()`: To convert the `rpy2` output back to Python.

In some cases (e.g., floats and strings) the conversion from Python to `rpy2` object happens automatically. In other cases (e.g., tuples and lists) the conversion needs to be made explicit through a call to `py_to_r()`. The table below gives the R object class corresponding to each relevant Python class, and indicates when to use the conversion function.

| Python type | R class | Conversion |
|------------------------|------------------------|------------------------|
| `int` | `integer` | *automatic* |
| `float` | `double` | *automatic* |
| `str` | `character` | *automatic* |
| `tuple`, `list` | `vector` | `py_to_r()` |
| `pandas.DataFrame`, `pandas.Series`, `numpy.ndarray` | `data.frame` | `py_to_r()` |
| `function` | `function` | *use a decorator (see below)* |

### Wrapper

The `healthiar` module contains the wrapper itself, allowing you to call any function available in the R package.

### Spatial data
Conversion of spatial data formats from Python (e.g., `geopandas`, `xarray`) to `rpy2` format is not possible. However, the module
`spatial` contains functions to read geographic vector or raster data in way that is compatible with the `healthiar` function `prepare_exposure()`, used to obtain tabular exposure data from geographic files:

- `read_raster()`: To read gridded data files (e.g., TIF);

- `read_vector`: To read geographic polygon files (e.g., GeoPackage).

## Examples
### Example 1: Typical case

This example illustrates a typical workflow, in this case comparing the health impact under two exposure scenarios. To start, let's make a call to `attribute_health()` for the reference scenario, after doing the necessary conversions of the input data:

In [ ]:
from healthiar.healthiar import healthiar
from healthiar.conversion import py_to_r, r_to_py

## convert Python lists to R vectors
exp_central_A = py_to_r([10, 18])
prop_pop_exp = py_to_r([0.7, 0.3])

## call healthiar function
scenario_A = healthiar.attribute_health(
    exp_central = exp_central_A,
    prop_pop_exp = prop_pop_exp,
    cutoff_central = 5,
    rr_central = 1.08,
    rr_increment = 10,
    erf_shape = "linear_log",
    bhd_central = 1200
)

Let's now calculate the health impact under a scenario where population exposure is halved, by passing the output of the previous call to `mod()` along with the alternative exposure values:

In [ ]:
exp_central_B = py_to_r([5, 9])

scenario_B = healthiar.attribute_mod(
    output_attribute = scenario_A, 
    exp_central = exp_central_B
)

The final step in the analysis is to calculate the hypothetical avoided impact, by passing both outputs to `compare()`:

In [ ]:
results_comparison = healthiar.compare(
    approach_comparison = "delta", # or "pif" (population impact fraction)
    output_attribute_scen_1 = scenario_A,
    output_attribute_scen_2 = scenario_B
)

We can now convert the end result back to Python:

In [ ]:
py_results_comparison = r_to_py(results_comparison)
#py_results_comparison["health_main"]["impact", "impact_scen_1", "impact_scen_2"].head()
py_results_comparison["health_main"][["impact", "impact_rounded", "impact_scen_1", "impact_scen_2", "bhd", "exp_type", "exp_scen_1", "exp_scen_2"]].head()

### Example 2: Function input
To pass a Python function as input to a function from `healthiar`, you need to wrap the `rpy2` function `rternalize()` around it using a
decorator. Let's take the `CubicSpline()` function from `scipy` as an example:

In [ ]:
import rpy2.rinterface as ri
from scipy.interpolate import CubicSpline

## define ERF with decorator ('@')
@ri.rternalize
def erf_fun(x):
    cs = CubicSpline(
      x = [0, 5, 10, 15, 20, 25, 30, 50, 70, 90, 110],
      y = [1.00, 1.04, 1.08, 1.12, 1.16, 1.20, 1.23, 1.35, 1.45, 1.53, 1.60]
    )
    return float(cs(x)[0])

## pass ERF to attribute_health()
results_pm_copd_mr_brt = healthiar.attribute_health(
  exp_central = 8.85,
  bhd_central = 30747,
  cutoff_central = 0,
  erf_eq_central = erf_fun
)

Note: The use of a decorator is only required when passing Python functions as `erf_eq_central`. Passing equations in the form of a string
requires no conversion.

### Example 3: Spatial input
Let's calculate a population-weighted average concentration of PM<sub>2.5</sub> for each region in Brussels, based on the population by municipality:

In [ ]:
from healthiar.spatial import read_raster, read_vector
from importlib.resources import files # to access package data files
import pathlib # to convert Windows path to POSIX path, if running Windows

path = files("healthiar.data")
exdat_pwm_1 = read_raster(path.joinpath("pm25.tif").as_posix())
exdat_pwm_2 = read_vector(path.joinpath("municipalities_brussels.gpkg").as_posix(), quiet = True)

population = [27335, 131426, 86534, 25425, 59840, 42638, 125734, 35369, 203105, 50060, 44735, 57993, 89204, 54077, 22901, 99096, 25630, 49704, 25441]

geo_id_macro = ["North", "North", "South", "South", "East", "East", "West", "East", "Center", "East", "North", "South", "South", "North", "West", "West", "West", "South", "West"]

pwm = healthiar.prepare_exposure(
  poll_grid = exdat_pwm_1, # Formal class SpatRaster,
  geo_units = exdat_pwm_2, # sf of the geographic sub-units
  population = py_to_r(population), # population per geographic sub-unit
  geo_id_macro = py_to_r(geo_id_macro)
)